# 州×商品カテゴリの売上構成

カテゴリ別GMVを比較し、州内の構成を調べる。カテゴリ注文数は同じ注文を複数カテゴリで数える場合がある。州の注文数は別の売上基準CSVから取得する。

共通の集計・作図関数は`../src/`に置き、Notebookには分析の問い、対象データ、結果の読み方を残す。


## データの確認

カテゴリの行数、欠損、重複を確認する。


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src").is_dir() and (path / "notebooks").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import seaborn as sns
from IPython.display import display
from src.project_paths import DATA_DIR, IMAGE_DIR
from src.sales_metrics import (
    aggregate_category_sales, state_totals as build_state_totals,
    category_state_metrics,
)
from src.analysis_plots import plot_state_category_bars, plot_category_gmv_share

df_category = pd.read_csv(DATA_DIR / "sales_by_category_state.csv")
df_category.head()

In [2]:
df_category.shape

(1388, 6)

In [3]:
df_category.dtypes

customer_state                str
product_category              str
category_order_count        int64
order_item_line_count       int64
category_gmv              float64
category_gmv_per_order    float64
dtype: object

In [4]:
df_category.isnull().sum()

customer_state            0
product_category          0
category_order_count      0
order_item_line_count     0
category_gmv              0
category_gmv_per_order    0
dtype: int64

In [5]:
df_category.duplicated().sum()

np.int64(0)

In [6]:
df_category.describe().T

,count,mean,std,min,25%,50%,75%,max
category_order_count,1388.0,70.083573,268.113458,1.0,2.00,8.000000,33.000000,4347.00
order_item_line_count,1388.0,79.392651,308.366898,1.0,3.00,8.000000,37.000000,5157.00
category_gmv,1388.0,9525.575007,33185.958310,6.8,311.15,1309.380000,5764.947500,472238.07
category_gmv_per_order,1388.0,195.974273,246.016877,6.8,90.00,132.529687,200.784233,2919.40


In [7]:
df_category.groupby("customer_state")[[
    "category_order_count", 
    "order_item_line_count", 
    "category_gmv", 
    "category_gmv_per_order"
    ]].describe()

category_order_count                                            \
                              count        mean         std  min   25%    50%   
customer_state                                                                  
AC                             28.0    2.892857    2.096545  1.0   1.0    2.0   
AL                             43.0    9.232558   12.424701  1.0   1.0    4.0   
AM                             38.0    3.815789    3.797974  1.0   1.0    2.0   
AP                             21.0    3.190476    2.315579  1.0   2.0    2.0   
BA                             63.0   52.079365   76.126868  1.0   5.0   11.0   
CE                             53.0   24.283019   33.865331  1.0   2.0    7.0   
DF                             61.0   34.295082   51.515805  1.0   4.0    7.0   
ES                             59.0   34.016949   48.537398  1.0   3.0   10.0   
GO                             63.0   31.428571   48.101946  1.0   3.0    6.0   
MA                             47.0   15.340426   19.988341  1.0   1.0    5.0   
MG                             71.0  161.366197  263.265387  1.0  10.0   28.0   
MS                             55.0   12.800000   16.408444  1.0   1.5    4.0   
MT                             54.0   16.555556   21.269756  1.0   2.0    5.5   
PA                             52.0   18.288462   24.464730  1.0   1.0    4.5   
PB                             47.0   11.021277   14.379166  1.0   1.0    4.0   
PE                             57.0   28.052632   43.685737  1.0   3.0    6.0   
PI                             48.0    9.979167   11.997322  1.0   2.0    4.0   
PR                             65.0   76.353846  114.433309  1.0   7.0   14.0   
RJ                             71.0  175.633803  286.359626  1.0  10.5   40.0   
RN                             46.0   10.369565   12.438933  1.0   1.0    4.5   
RO                             42.0    5.833333    5.716543  1.0   1.0    4.5   
RR                             17.0    2.411765    1.543487  1.0   1.0    2.0   
RS                             68.0   79.514706  126.953922  1.0   5.0   17.0   
SC                             65.0   54.969231   82.567769  1.0   4.0   13.0   
SE                             42.0    8.000000    9.730515  1.0   1.0    3.0   
SP                             73.0  559.301370  943.379235  2.0  32.0  109.0   
TO                             39.0    7.051282    7.772924  1.0   1.0    3.0   

                               order_item_line_count              ...  \
                   75%     max                 count        mean  ...   
customer_state                                                    ...   
AC                4.00     9.0                  28.0    3.250000  ...   
AL               12.00    62.0                  43.0    9.930233  ...   
AM                4.75    15.0                  38.0    4.289474  ...   
AP                4.00    10.0                  21.0    3.857143  ...   
BA               71.50   319.0                  63.0   58.460317  ...   
CE               40.00   152.0                  53.0   26.905660  ...   
DF               43.00   219.0                  61.0   38.606557  ...   
ES               53.50   197.0                  59.0   37.711864  ...   
GO               43.00   206.0                  63.0   36.142857  ...   
MA               21.50    82.0                  47.0   17.021277  ...   
MG              198.50  1120.0                  71.0  181.915493  ...   
MS               20.00    60.0                  55.0   14.745455  ...   
MT               23.50    83.0                  54.0   19.203704  ...   
PA               27.50    99.0                  52.0   20.269231  ...   
PB               15.50    75.0                  47.0   12.468085  ...   
PE               41.00   220.0                  57.0   30.631579  ...   
PI               14.25    50.0                  48.0   10.895833  ...   
PR              107.00   420.0                  65.0   86.907692  ...   
RJ              191.00  1357.0                  71.

In [8]:

for column in ["category_order_count", "order_item_line_count", "category_gmv", "category_gmv_per_order"]:
    max_index = df_category[column].idxmax()
    print(f"\n{column}")
    display(df_category.loc[max_index, ["customer_state", "product_category", column]])




category_order_count


customer_state                      SP
product_category        bed_bath_table
category_order_count              4347
Name: 1276, dtype: object


order_item_line_count


customer_state                       SP
product_category         bed_bath_table
order_item_line_count              5157
Name: 1276, dtype: object


category_gmv


customer_state                  SP
product_category    bed_bath_table
category_gmv             472238.07
Name: 1276, dtype: object


category_gmv_per_order


customer_state                                MT
product_category          signaling_and_security
category_gmv_per_order                    2919.4
Name: 620, dtype: object

In [9]:
weighted_average_category_order_value = (
    df_category["category_gmv"].sum() / df_category["category_order_count"].sum()
)

weighted_ave_order_item_line_count = (
    df_category["category_gmv"].sum() / df_category["order_item_line_count"].sum()
)

print(weighted_average_category_order_value)
print(weighted_ave_order_item_line_count)

135.91737026604713
119.98056308247955


## カテゴリと州の集計

比率列は足さず、金額と件数の合計から求め直す。


In [ ]:
# 各カテゴリのGMVと件数を合計してから比率を計算する。
# 元の「category_gmv_per_order」を合計しない。
df_product_category = aggregate_category_sales(df_category, "product_category")
display(df_product_category)

In [ ]:
# この表のカテゴリ注文数は重複を含む。州のユニーク注文数ではない。
df_state = aggregate_category_sales(df_category, "customer_state")
display(df_state)

In [ ]:
# 以下の注文シェアは「カテゴリを含む注文数の延べ件数」に占める割合。
# 州のユニーク注文数の割合は売上基準Notebookを使う。
df_state["order_share_pct"] = (
    df_state["category_order_count"] / df_state["category_order_count"].sum() * 100
)
df_state["order_line_pct"] = (
    df_state["order_item_line_count"] / df_state["order_item_line_count"].sum() * 100
)
df_state["gmv_share_pct"] = df_state["category_gmv"] / df_state["category_gmv"].sum() * 100
display(df_state.head(10).round(2))

カテゴリGMVの州別合計は売上基準と比較できる。一方、カテゴリ注文数の合計には同じ注文の重複があり、州のユニーク注文数とは異なる。
続いて、上位３州でなんのカテゴリーがgmvに貢献しているのかを調べる。

In [ ]:
# 州×カテゴリのGMVを確認。カテゴリ注文の平均額は州AOVではない。
df_major_category = aggregate_category_sales(
    df_category, ["customer_state", "product_category"]
)
target_states = ["SP", "RJ", "MG"]
df_selected = df_major_category.loc[
    df_major_category["customer_state"].isin(target_states)
].copy()
df_selected["customer_state"] = pd.Categorical(
    df_selected["customer_state"], categories=target_states, ordered=True
)
display(df_selected.sort_values(
    ["customer_state", "category_gmv"], ascending=[True, False]
))

## 商品カテゴリ構成の図

全体と各州で同じカテゴリの色をそろえる。


In [ ]:
# 全体GMV上位カテゴリの色を州別棒グラフでも共通にする。
top10_categories = df_product_category.head(10)["product_category"].tolist()
category_color_map = dict(zip(
    top10_categories, sns.color_palette("Blues_r", n_colors=10).as_hex()
))
category_color_map["others"] = "#cacdd1"
plot_state_category_bars(df_category, ["SP", "RJ", "MG"], category_color_map)

In [ ]:
# 全体のカテゴリGMV構成。カテゴリ別注文数は重複するため使わない。
fig = plot_category_gmv_share(df_category, category_color_map)

In [ ]:
# AL・PAも同じ色の規則で描く。
plot_state_category_bars(df_category, ["AL", "PA"], category_color_map)

## 州全体の注文数を分母にした比較

カテゴリ注文数の合計には重複がある。


In [ ]:
# 州全体の注文数は月×州の基準表から取り、カテゴリ注文率を計算する。
df_baseline = pd.read_csv(DATA_DIR / "sales_baseline_by_month_state.csv")
category_comparison = category_state_metrics(df_category, df_baseline)
target_states = ["SP", "AL", "PA"]
target_categories = ["bed_bath_table", "toys"]
display(category_comparison.loc[
    category_comparison["customer_state"].isin(target_states)
    & category_comparison["product_category"].isin(target_categories),
    ["customer_state", "product_category", "state_orders", "category_order_count",
     "category_order_rate_pct", "category_gmv_share_pct"],
].sort_values(["product_category", "customer_state"]).round(2))

### SP・RJ・MGのカテゴリ構成と平均注文額の比較

注文数とGMVが大きい3州を、平均注文額が高かったPB・APとも比較する。比較するカテゴリは、前の分析で差が目立ったものと、3州でGMVが大きい`bed_bath_table`を選んだ。州全体の注文数を分母として、カテゴリ購入割合と平均注文額への構成額を計算する。

- カテゴリ購入割合 = カテゴリを含む注文数 ÷ 州のユニークな注文数
- カテゴリ注文あたりGMV = カテゴリGMV ÷ カテゴリを含む注文数（注文全体の金額ではない）
- 平均注文額への構成額 = カテゴリGMV ÷ 州のユニークな注文数

これらは過去の購入構成を示す。施策による平均注文額やGMVの変化は示さない。


In [ ]:
# SP・RJ・MGを売上規模の大きい州、PB・APを参考州として読む。
state_totals = build_state_totals(df_baseline)
major_states = ["SP", "RJ", "MG"]
reference_states = ["PB", "AP"]
comparison_states = major_states + reference_states
display(state_totals.loc[
    state_totals["customer_state"].isin(comparison_states),
    ["customer_state", "state_orders", "state_gmv", "average_order_value"],
].sort_values("state_gmv", ascending=False).round(2))

In [ ]:
# 州全体の注文数を分母に、同じカテゴリの割合とAOV構成額を比べる。
focus_categories = [
    "bed_bath_table", "health_beauty", "watches_gifts",
    "computers_accessories", "computers", "auto",
]
major_category_comparison = category_comparison.loc[
    category_comparison["customer_state"].isin(comparison_states)
    & category_comparison["product_category"].isin(focus_categories)
].copy()
major_category_comparison["customer_state"] = pd.Categorical(
    major_category_comparison["customer_state"],
    categories=comparison_states, ordered=True,
)
display(major_category_comparison.sort_values(
    ["product_category", "customer_state"]
)[[
    "product_category", "customer_state", "state_orders", "category_order_count",
    "category_order_rate_pct", "category_gmv_per_category_order", "aov_part",
]].round(2))

### SP・RJ・MGそれぞれのカテゴリGMV構成

全体のカテゴリ円グラフと同じ形式で、各州のカテゴリGMV上位10件を表示し、残りを`others`にまとめる。各円グラフの分母は**その州のGMV**。円グラフ同士の大きさは州のGMV規模を表さないため、上の州別GMV表とあわせて読む。


In [ ]:
# 全体図のカテゴリ色を使い、上位10カテゴリとその他を州ごとに表示する。
for state in major_states:
    fig = plot_category_gmv_share(
        df_category, category_color_map, state=state,
        image_path=IMAGE_DIR / f"Top10_GMV_Share_by_Product_Category_in_{state}.png",
    )

### 次に読むこと

- SP・RJ・MGで同じカテゴリでも購入割合とカテゴリ注文あたりGMVは異なるか。
- 上位カテゴリの件数は十分か。PB・APの少数注文を3州にそのまま当てはめない。
- 月別の安定性、商品・販売者の構成、販促費と利益を確認するまでは、客単価向上策の効果を断定しない。
